# 10 – Models: does a model trained before the 2018 change still work after it?
**Input:** `model_data.csv` (from `09_Model_Data.ipynb`)

We train three classification models from the course – **Logistic Regression** (Chapter 12), **Decision Tree** (Chapter 9) and **K-Nearest Neighbors** (Chapter 12) – and test them in three experiments:

| Experiment | Train on | Test on |
| --- | --- | --- |
| **H2** (main question) | 80% of Pre | the other 20% of Pre **and** Post |
| **H3** (placebo) | 80% of Pre before 1 Jan 2017 | the other 20% of it **and** Pre after 1 Jan 2017 |
| **Reference** (random split) | random 80% of Pre + Post | the other 20% |

Because only about 1 of 5 inspections fails, we measure **accuracy, sensitivity, specificity, PPV** and the **AUC** (area under the ROC curve), not only accuracy (Chapter 12).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

pandas, numpy and matplotlib as always; from scikit-learn the train/test split and KFold (Chapter 10), StandardScaler for the scaling, the three models, and the metrics: the confusion matrix (TP, FP, TN, FN), the ROC curve and its AUC (Chapter 12).

In [ ]:
data = pd.read_csv('model_data.csv')
id_columns = ['Inspection ID', 'Inspection Date', 'Period', 'Placebo_Period', 'Fail']
features = [c for c in data.columns if c not in id_columns]
print(data.shape)
print('Number of features:', len(features))

(Write the shape and the number of features here: 88 columns, 83 of them are features – 12 numeric and 71 dummies.)

## 1. Helper functions

In [ ]:
def evaluate(y_true, prob, cutoff=0.5):
    # prob = predicted probability of Fail; above the cutoff we predict Fail
    y_pred = (prob >= cutoff).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {'Accuracy': (tp + tn) / (tp + tn + fp + fn),
            'Sensitivity': tp / (tp + fn),
            'Specificity': tn / (tn + fp),
            'PPV': tp / (tp + fp) if (tp + fp) > 0 else 0,
            'AUC': roc_auc_score(y_true, prob)}

`evaluate` calculates the metrics of Chapter 12 from the confusion matrix (Positive = **Fail**):
- **Accuracy** = (TP + TN) / all
- **Sensitivity** = TP / (TP + FN): of the inspections that really failed, how many did the model find? (the most important one for us: a missed failing restaurant is the dangerous mistake)
- **Specificity** = TN / (TN + FP): of the inspections that passed, how many did the model call "Pass"?
- **PPV** = TP / (TP + FP): when the model says "Fail", how often is it right?
- **AUC**: area under the ROC curve, one number for all cutoffs (0.5 = random guessing, 1 = perfect).

In [ ]:
def make_models(knn_k, tree_depth):
    return {'Logistic Regression': LogisticRegression(max_iter=1000),
            'Decision Tree': DecisionTreeClassifier(max_depth=tree_depth, random_state=42),
            'KNN': KNeighborsClassifier(n_neighbors=knn_k)}

def run_experiment(name, train, tests, knn_k, tree_depth):
    # scaling: fitted on the training data only, then applied to every test set (no leakage)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[features])
    y_train = train['Fail']
    rows = []
    for model_name, model in make_models(knn_k, tree_depth).items():
        model.fit(X_train, y_train)
        for test_name, test in tests.items():
            prob = model.predict_proba(scaler.transform(test[features]))[:, 1]
            result = evaluate(test['Fail'], prob)
            result.update({'Experiment': name, 'Model': model_name, 'Tested on': test_name})
            rows.append(result)
    return pd.DataFrame(rows)[['Experiment', 'Model', 'Tested on', 'Accuracy', 'Sensitivity', 'Specificity', 'PPV', 'AUC']]

`make_models` creates the three models. `run_experiment` scales the features with **StandardScaler** (mean 0, std 1), fitted on the **training data only** and then used on the test sets, so no information from the test data leaks into training. Scaling is needed for KNN, which uses distances (otherwise Days_Since_First_Seen, in thousands, would dominate Prev_Fail, 0 or 1 – Chapter 11); logistic regression also trains better with scaled features, and the decision tree is not affected. Then every model is trained once and tested on every test set.

## 2. Split the data

In [ ]:
pre = data[data['Period'] == 'Pre']
post = data[data['Period'] == 'Post']

# H2: 80% of Pre for training, 20% of Pre and all of Post for testing
pre_train, pre_test = train_test_split(pre, test_size=0.2, random_state=42)

# H3 (placebo): the same, with 1 January 2017 as a fake change date inside Pre
before = pre[pre['Placebo_Period'] == 'Before 2017']
after = pre[pre['Placebo_Period'] == 'After 2017']
before_train, before_test = train_test_split(before, test_size=0.2, random_state=42)

# Reference: random 80/20 split of Pre and Post together (years mixed)
pre_post = pd.concat([pre, post])
random_train, random_test = train_test_split(pre_post, test_size=0.2, random_state=42)

for name, part in [('Pre train', pre_train), ('Pre test', pre_test), ('Post', post),
                   ('Before 2017 train', before_train), ('Before 2017 test', before_test), ('After 2017', after),
                   ('Random train', random_train), ('Random test', random_test)]:
    print(f"{name:18s} {len(part):7d} rows   fail rate {part['Fail'].mean():.3f}")

(Write the sizes and fail rates here.) In each experiment the model is tested **twice**: on unseen inspections from its **own** period (20% held out) and on the **later** period. The difference between the two results is the drop caused by moving to a later period. random_state=42 keeps the splits the same every time we run the notebook.

## 3. Choose the model settings with k-fold cross-validation (Chapter 10)

In [ ]:
def cv_auc(model, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for train_index, val_index in kf.split(X):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X.iloc[train_index])
        X_val = scaler.transform(X.iloc[val_index])
        model.fit(X_tr, y.iloc[train_index])
        prob = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y.iloc[val_index], prob))
    return np.mean(scores)

# a sample of 10,000 Pre training rows: KNN is slow, because every prediction compares with all training rows
sample = pre_train.sample(10000, random_state=42)
X_s, y_s = sample[features], sample['Fail']

for depth in [3, 5, 8, 12]:
    print(f'Decision tree, max_depth = {depth:2d}: mean AUC = {cv_auc(DecisionTreeClassifier(max_depth=depth, random_state=42), X_s, y_s):.3f}')
print()
for k in [5, 15, 25, 51]:
    print(f'KNN, k = {k:2d}: mean AUC = {cv_auc(KNeighborsClassifier(n_neighbors=k), X_s, y_s):.3f}')

5-fold cross-validation **inside the Pre training data only** (the test sets are not used). A tree that is too deep and a KNN with a very small k overfit (Chapter 10); a very shallow tree or a very large k underfit. We pick the value with the highest mean AUC. We use a sample of 10,000 rows because KNN is slow (Chapter 12: "predictions can take a long time"); k is odd so there are no ties in the vote. (Write the results here and change `best_depth` and `best_k` in the next cell to the best values.)

In [ ]:
best_depth = 5   # change to the best max_depth above
best_k = 25      # change to the best k above

(Write here which values were chosen and why.)

## 4. H2 – train on Pre, test on Pre and on Post

In [ ]:
h2 = run_experiment('H2', pre_train, {'Pre (same period)': pre_test, 'Post (after change)': post}, best_k, best_depth)
h2.round(3)

(Write the results here: for each model compare the Pre and the Post row, especially sensitivity and AUC.)

## 5. H3 – placebo: train before 2017, test before and after 1 January 2017

In [ ]:
h3 = run_experiment('H3', before_train, {'Before 2017 (same period)': before_test, 'After 2017 (placebo)': after}, best_k, best_depth)
h3.round(3)

(Write the results here.)

## 6. Reference – random split of Pre and Post together

In [ ]:
ref = run_experiment('Reference', random_train, {'Random 20%': random_test}, best_k, best_depth)
ref.round(3)

(Write the results here: does the random split look better than the Pre → Post test?)

## 7. Compare the drops: real change (2018) vs placebo (2017)

In [ ]:
def drop_table(results, same, later):
    a = results[results['Tested on'] == same].set_index('Model')
    b = results[results['Tested on'] == later].set_index('Model')
    return (b[['Accuracy', 'Sensitivity', 'Specificity', 'PPV', 'AUC']] - a[['Accuracy', 'Sensitivity', 'Specificity', 'PPV', 'AUC']])

drops = pd.concat({'2018 change (H2)': drop_table(h2, 'Pre (same period)', 'Post (after change)'),
                   '2017 placebo (H3)': drop_table(h3, 'Before 2017 (same period)', 'After 2017 (placebo)')})
drops.round(3)

Change of every metric when the model moves from its own period to the later period (negative = worse). **H2:** if the 2018 drop is clearly larger than the 2017 placebo drop, the new food code made the model worse beyond the normal change over time. (Write the conclusion here.)

In [ ]:
for name, results in [('H2', h2), ('H3', h3)]:
    plt.figure(figsize=(7, 4))
    pivot = results.pivot(index='Model', columns='Tested on', values='AUC')
    pivot.plot.bar(ax=plt.gca(), rot=0)
    plt.title(f'AUC per model – {name}')
    plt.ylabel('AUC')
    plt.ylim(0.5, 1)
    plt.legend(title='Tested on')
    plt.show()

The AUC of each model on its own period and on the later period, for the real change (H2) and the placebo (H3). The y-axis starts at 0.5 (random guessing), so be careful: the bars make small differences look bigger (Chapter 12, "look carefully at the axes"). (Write what you see here.)

## 8. ROC curves of the logistic regression (H2)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(pre_train[features])
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, pre_train['Fail'])

plt.figure(figsize=(6, 5))
for test_name, test in [('Pre (same period)', pre_test), ('Post (after change)', post)]:
    prob = lr.predict_proba(scaler.transform(test[features]))[:, 1]
    fpr, tpr, cut = roc_curve(test['Fail'], prob)
    plt.plot(fpr, tpr, label=f'{test_name}, AUC = {roc_auc_score(test["Fail"], prob):.3f}')
plt.plot([0, 1], [0, 1], 'k--', label='Random guessing')
plt.xlabel('1 - Specificity (false positive rate)')
plt.ylabel('Sensitivity (true positive rate)')
plt.title('ROC curve – logistic regression trained on Pre')
plt.legend()
plt.show()

The ROC curve shows sensitivity against 1 − specificity for **every** cutoff (Chapter 12). The closer the curve is to the top-left corner, the better; the dashed line is random guessing. (Write here how far apart the Pre and Post curves are.)

## 9. Changing the cutoff (Chapter 12)

In [ ]:
rows = []
for cutoff in [0.1, 0.2, 0.3, 0.4, 0.5]:
    for test_name, test in [('Pre', pre_test), ('Post', post)]:
        prob = lr.predict_proba(scaler.transform(test[features]))[:, 1]
        result = evaluate(test['Fail'], prob, cutoff)
        result.update({'Cutoff': cutoff, 'Tested on': test_name})
        rows.append(result)
pd.DataFrame(rows)[['Cutoff', 'Tested on', 'Accuracy', 'Sensitivity', 'Specificity', 'PPV']].round(3)

With the usual cutoff 0.5 the model says "Fail" only when it is quite sure; because only about 22% of the inspections fail, it then misses many failing restaurants (low sensitivity). A lower cutoff finds more of them (higher sensitivity) but also sends inspectors to more clean places (lower specificity) – the trade-off of Chapter 12. Because a missed failing restaurant is the dangerous mistake, a lower cutoff can make sense. (Write here which cutoff gives a good balance, and if the Pre → Post drop is the same at every cutoff.)

## 10. What did the models learn?

In [ ]:
weights = pd.Series(lr.coef_[0], index=features).sort_values()
print('Strongest negative weights (less likely to fail):')
print(weights.head(8).round(3))
print()
print('Strongest positive weights (more likely to fail):')
print(weights.tail(8).round(3))

The weights of the logistic regression (scaled features, so they can be compared with each other). A positive weight means the feature goes together with failing, a negative weight with passing (Chapter 12). Be careful: the features are often correlated (for example Prev_Fail and Past_Fail_Rate), so we should not read too much into one single weight. (Write the most important features here.)

In [ ]:
tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
tree.fit(pre_train[features], pre_train['Fail'])

plt.figure(figsize=(18, 7))
plot_tree(tree, max_depth=2, feature_names=features, class_names=['Pass', 'Fail'], filled=True, fontsize=9)
plt.title('Top of the decision tree trained on Pre (first 3 levels)')
plt.show()

The first levels of the decision tree (Chapter 9): each box is a question (rule) about one feature, and the data is split into "yes" (left) and "no" (right). The tree does not need scaling, so it is trained on the original values and the rules are easy to read. (Write here which questions the tree asks first.)